
# Projeto 14 — Violência Contra Mulheres: Registros de Feminicídios por Estado (2015–2024)

**Disciplina:** Linguagem de Programação — Análise e Visualização de Dados com Python  
**Objetivo do projeto:** Mapear os registros de feminicídio por estado, analisando evolução temporal, comparação proporcional entre UFs e possíveis implicações para políticas públicas.

---
## Estrutura do notebook
1. Título e introdução  
2. Definição do problema  
3. Definição do KPI  
4. Aquisição dos dados  
5. Limpeza e preparação dos dados  
6. Análise exploratória  
7. Visualizações  
8. Insights e interpretação  
9. Conclusão  
10. Reflexão final (data storytelling)



## 1. Introdução

A violência contra as mulheres é um problema social grave e persistente. Entre suas formas mais extremas está o feminicídio, que exige atenção especial em políticas públicas de prevenção, proteção e justiça.

Neste projeto, vamos investigar os registros de feminicídios por estado entre 2015 e 2024, buscando identificar padrões temporais e estados mais críticos.



## 2. Definição do problema

**Pergunta de análise:**  
Como os registros de feminicídios evoluíram entre 2015 e 2024 nos estados brasileiros e quais UFs apresentam maior gravidade proporcional do problema?

**Por que essa pergunta é relevante?**  
Porque ajuda a identificar estados mais vulneráveis, acompanhar a evolução do problema e apoiar políticas públicas voltadas à proteção das mulheres.

**Qual decisão poderia ser tomada com base nessa análise?**  
Direcionar ações de prevenção, reforço da rede de proteção e campanhas de conscientização para estados com maiores taxas e piores tendências.



## 3. Definição do KPI

### KPI principal
**Taxa de feminicídios por 100 mil mulheres**

### Fórmula
\[
	ext{Taxa de Feminicídios} = rac{	ext{Número de Feminicídios}}{	ext{População Feminina}} 	imes 100000
\]

### Justificativa
Esse indicador permite comparar estados de forma proporcional, evitando distorções causadas pelo tamanho da população feminina.

### KPI complementar
- Total de feminicídios por estado
- Evolução anual da taxa
- Estados com maior crescimento ou redução ao longo do tempo


In [ ]:

# 4. Aquisição dos dados

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_theme(style="whitegrid")

# Leitura da base
df = pd.read_excel("Simulacao_Feminicidio_Estados_2015_2024.xlsx")

# Visualização inicial
df.head()



### Descrição das variáveis principais
- **Ano**: ano de referência
- **Estado**: unidade da federação
- **Feminicídios**: total de registros
- **População Feminina**: população feminina estimada
- **Taxa por 100 mil mulheres**: indicador proporcional do problema


In [ ]:

# 5. Limpeza e preparação dos dados

print("Dimensões da base:", df.shape)
print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

print("\nQuantidade de linhas duplicadas:", df.duplicated().sum())


In [ ]:

# Ajuste defensivo de nomes de colunas
rename_map = {
    "Taxa por 100 mil mulheres": "Taxa_Feminicidio",
    "Taxa de Feminicídios por 100 mil mulheres": "Taxa_Feminicidio",
    "Feminicidios": "Feminicidios",
    "Populacao Feminina": "Populacao_Feminina"
}
df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

# Se a taxa não existir na base, calcular
if "Taxa_Feminicidio" not in df.columns and {"Feminicidios", "Populacao_Feminina"}.issubset(df.columns):
    df["Taxa_Feminicidio"] = (df["Feminicidios"] / df["Populacao_Feminina"]) * 100000

# Garantindo tipos corretos
df["Ano"] = df["Ano"].astype(int)
df["Estado"] = df["Estado"].astype(str)
df["Taxa_Feminicidio"] = df["Taxa_Feminicidio"].astype(float)

df.head()



### Comentário sobre a preparação
Nesta etapa:
- verificamos a qualidade da base;
- ajustamos nomes de colunas, quando necessário;
- garantimos o tipo correto das variáveis;
- calculamos a taxa por 100 mil mulheres, caso ela não estivesse pronta.


In [ ]:

# 6. Análise exploratória

# Estatísticas descritivas da taxa
df["Taxa_Feminicidio"].describe()


In [ ]:

# Taxa média por ano
taxa_por_ano = df.groupby("Ano")["Taxa_Feminicidio"].mean().reset_index()
taxa_por_ano


In [ ]:

# Taxa média por estado no período
taxa_por_estado = (
    df.groupby("Estado")["Taxa_Feminicidio"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)
taxa_por_estado


In [ ]:

# Tabela dinâmica: taxa por ano e por estado
taxa_estado_ano = df.pivot_table(
    index="Ano",
    columns="Estado",
    values="Taxa_Feminicidio",
    aggfunc="mean"
)

taxa_estado_ano


In [ ]:

# Variação percentual da taxa por estado ao longo do tempo
df = df.sort_values(["Estado", "Ano"])
df["Variacao_Percentual"] = df.groupby("Estado")["Taxa_Feminicidio"].pct_change() * 100

df.head(10)



## 7. Visualizações

Os gráficos abaixo ajudam a responder:
- quais estados apresentam maiores taxas;
- como o problema evolui ao longo do tempo;
- em quais períodos a situação foi mais crítica.


In [ ]:

# Gráfico 1: evolução da taxa média ao longo do tempo
plt.figure(figsize=(10, 6))
sns.lineplot(data=taxa_por_ano, x="Ano", y="Taxa_Feminicidio", marker="o")
plt.title("Evolução da Taxa Média de Feminicídios por 100 mil Mulheres (2015–2024)")
plt.xlabel("Ano")
plt.ylabel("Taxa de Feminicídios")
plt.xticks(taxa_por_ano["Ano"], rotation=45)
plt.show()


In [ ]:

# Gráfico 2: ranking dos estados por taxa média
plt.figure(figsize=(10, 6))
sns.barplot(data=taxa_por_estado, x="Estado", y="Taxa_Feminicidio")
plt.title("Taxa Média de Feminicídios por Estado (2015–2024)")
plt.xlabel("Estado")
plt.ylabel("Taxa de Feminicídios")
plt.xticks(rotation=45)
plt.show()


In [ ]:

# Gráfico 3: evolução da taxa por estado
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x="Ano", y="Taxa_Feminicidio", hue="Estado", legend=False)
plt.title("Evolução da Taxa de Feminicídios por Estado")
plt.xlabel("Ano")
plt.ylabel("Taxa de Feminicídios")
plt.xticks(sorted(df["Ano"].unique()), rotation=45)
plt.show()


In [ ]:

# Gráfico 4: distribuição das taxas
plt.figure(figsize=(8, 5))
sns.histplot(df["Taxa_Feminicidio"], bins=12, kde=True)
plt.title("Distribuição da Taxa de Feminicídios")
plt.xlabel("Taxa de Feminicídios")
plt.ylabel("Frequência")
plt.show()



## 8. Insights e interpretação

Nesta etapa, o mais importante é explicar o que os dados indicam sobre gravidade, tendência e desigualdade entre estados.


In [ ]:

# Estado com maior taxa média no período
estado_mais_critico = taxa_por_estado.loc[taxa_por_estado["Taxa_Feminicidio"].idxmax()]
estado_mais_critico


In [ ]:

# Estado com menor taxa média no período
estado_menos_critico = taxa_por_estado.loc[taxa_por_estado["Taxa_Feminicidio"].idxmin()]
estado_menos_critico


In [ ]:

# Ano com maior taxa média nacional
ano_mais_critico = taxa_por_ano.loc[taxa_por_ano["Taxa_Feminicidio"].idxmax()]
ano_mais_critico



### Modelo de interpretação esperada

**Insight 1 — Diferenças entre estados**  
A comparação entre os estados mostra que o feminicídio não se distribui de forma homogênea no país. Algumas UFs apresentam taxas proporcionalmente maiores, o que indica necessidade de atenção prioritária.

**Insight 2 — Evolução temporal**  
A série histórica permite observar se o problema está se agravando, se apresenta estabilidade ou se houve melhora em determinados períodos. Essa análise é importante para avaliar a efetividade de ações de prevenção e proteção.

**Insight 3 — Gravidade proporcional**  
O uso da taxa por 100 mil mulheres é essencial para evitar interpretações distorcidas. Estados com população maior podem ter mais casos absolutos, mas não necessariamente maior gravidade proporcional.

**Insight 4 — Limitações da análise**  
A base permite comparar estados e anos, mas não explica sozinha os fatores sociais, culturais, institucionais e econômicos que influenciam a violência contra a mulher. Uma análise mais aprofundada exigiria integração com outras variáveis.



## 9. Conclusão

Com base na análise realizada, é possível identificar quais estados apresentam maior taxa de feminicídios e como esse indicador evolui ao longo do tempo.

Esses resultados ajudam a mapear áreas mais críticas e a orientar políticas públicas voltadas à proteção das mulheres, fortalecimento da rede de apoio e prevenção da violência.

**Resposta à pergunta inicial:**  
Os registros de feminicídio variaram entre os estados e ao longo do tempo, com algumas UFs se destacando negativamente por apresentarem taxas proporcionalmente mais elevadas.



## 10. Reflexão Final (Data Storytelling)

Se este projeto fosse apresentado a um gestor, a principal mensagem seria:

> “A violência letal contra as mulheres não afeta todos os estados da mesma forma. Há unidades da federação que exigem maior atenção em políticas públicas de proteção, prevenção e monitoramento.”

### Ação recomendada
- Reforçar a rede de proteção nos estados com maior taxa;
- Ampliar campanhas de conscientização e prevenção;
- Integrar a análise com dados sobre atendimento, denúncias e medidas protetivas em estudos futuros.



## Sugestões de aprofundamento (opcional)
- Comparar casos absolutos e taxa proporcional;
- Analisar crescimento ou redução por estado;
- Construir ranking anual de UFs;
- Transformar a análise em dashboard interativo em etapa futura.
